# Phase 0 Webcam Landmark Normalization Probe

This notebook is Colab-friendly and uses browser-based webcam capture.

What it checks:
- MediaPipe Face Landmarker runs on a webcam frame
- local landmark normalization produces stable scale/centroid stats
- moving your face around the frame does not wildly change the normalized output

Notes:
- Colab does not reliably support `cv2.VideoCapture(0)` for a live local webcam.
- This notebook captures one browser webcam frame at a time. Re-run the probe cell after moving your head or body.
- Upload a MediaPipe Face Landmarker `.task` file when prompted, or set `MODEL_PATH` to a file in Drive.

In [ ]:
!pip -q install numpy opencv-python mediapipe


In [ ]:
from base64 import b64decode
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np
from google.colab.output import eval_js
from google.colab.patches import cv2_imshow
from IPython.display import Javascript, display


DEFAULT_ANCHOR_INDICES = (1, 6, 168)
DEFAULT_SCALE_INDICES = (33, 263)


def _as_landmark_array(landmarks):
    array = np.asarray(landmarks, dtype=np.float32)
    if array.ndim != 2 or array.shape[1] != 3:
        raise ValueError(f"Expected landmark array shaped (n, 3), got {array.shape}")
    return array


def normalize_landmarks(landmarks, anchor_indices=DEFAULT_ANCHOR_INDICES, scale_indices=DEFAULT_SCALE_INDICES):
    array = _as_landmark_array(landmarks)
    anchor_points = array[list(anchor_indices)]
    anchor = anchor_points.mean(axis=0)
    scale_points = array[list(scale_indices)]
    if len(scale_points) < 2:
        raise ValueError("Scale estimation needs at least two landmark indices")
    scale = float(np.linalg.norm(scale_points[0] - scale_points[1]))
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0
    normalized = (array - anchor) / scale
    return normalized, {"anchor": anchor, "scale": scale}


def capture_frame(filename: str = "capture.jpg", quality: float = 0.8) -> Path:
    js = Javascript(r'''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => setTimeout(resolve, 1000));
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      video.remove();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js(f"takePhoto({quality})")
    binary = b64decode(data.split(',')[1])
    path = Path(filename)
    path.write_bytes(binary)
    return path


In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Upload a MediaPipe Face Landmarker .task file to continue.")

MODEL_PATH = Path(next(iter(uploaded.keys())))
print(f"Using model: {MODEL_PATH}")


In [ ]:
base_options = mp.tasks.BaseOptions(model_asset_path=str(MODEL_PATH))
options = mp.tasks.vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
    num_faces=1,
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
)


def probe_once(landmarker, filename: str = "capture.jpg") -> None:
    image_path = capture_frame(filename)
    frame = cv2.imread(str(image_path))
    if frame is None:
        raise RuntimeError(f"Unable to read captured frame: {image_path}")

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    result = landmarker.detect(mp_image)

    if result.face_landmarks:
        landmarks = result.face_landmarks[0]
        point_array = [[point.x, point.y, point.z] for point in landmarks]
        normalized, stats = normalize_landmarks(point_array)
        centroid = normalized.mean(axis=0)
        spread = normalized.std(axis=0)
        cv2.putText(
            frame,
            f"scale={stats['scale']:.4f} centroid={np.linalg.norm(centroid):.4f} spread={spread.mean():.4f}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2,
        )
        print({
            "scale": float(stats["scale"]),
            "centroid_norm": float(np.linalg.norm(centroid)),
            "spread_mean": float(spread.mean()),
        })
    else:
        print("No face detected in this capture.")

    cv2_imshow(frame)


with mp.tasks.vision.FaceLandmarker.create_from_options(options) as landmarker:
    probe_once(landmarker)

print("Re-run the last cell after moving your face or body to compare the normalized output.")
